In [0]:
storage_account = 'salespipelineadls'
storage_key = 'YOUR_KEY_HERE'

spark.conf.set(
    f'fs.azure.account.key.{storage_account}.dfs.core.windows.net',
    storage_key
)

gold_revenue = f'abfss://gold@{storage_account}.dfs.core.windows.net/monthly_revenue/'
print('Config done.')

Config done.


In [0]:
from pyspark.sql import functions as F

df_gold = spark.read.format('delta').load(gold_revenue)
print(f'Gold rows to write: {df_gold.count()}')
display(df_gold.limit(5))

Gold rows to write: 96


order_year,order_month,region,total_revenue,order_count,avg_order_value,unique_customers,total_units_sold,report_month,load_ts
2022,1,East,5975106.00,90,66390.07,20,512,2022-01,2026-04-16T07:26:25.962Z
2022,1,North,2292276.50,51,44946.60,17,293,2022-01,2026-04-16T07:26:25.962Z
2022,1,South,4175829.00,47,88847.43,18,268,2022-01,2026-04-16T07:26:25.962Z
2022,1,West,3239373.00,30,107979.10,15,187,2022-01,2026-04-16T07:26:25.962Z
2022,2,East,5063232.50,97,52198.27,20,571,2022-02,2026-04-16T07:26:25.962Z


In [0]:
# Select only columns that exist in dbo.gold_monthly_revenue
# Column names must exactly match the SQL table
df_to_write = df_gold.select(
    F.col('report_month'),
    F.col('region'),
    F.col('total_revenue'),
    F.col('order_count'),
    F.col('avg_order_value'),
    F.col('load_ts').alias('load_date')
)

print(f'Columns to write: {df_to_write.columns}')
display(df_to_write.limit(5))

Columns to write: ['report_month', 'region', 'total_revenue', 'order_count', 'avg_order_value', 'load_date']


report_month,region,total_revenue,order_count,avg_order_value,load_date
2022-01,East,5975106.00,90,66390.07,2026-04-16T07:26:25.962Z
2022-01,North,2292276.50,51,44946.60,2026-04-16T07:26:25.962Z
2022-01,South,4175829.00,47,88847.43,2026-04-16T07:26:25.962Z
2022-01,West,3239373.00,30,107979.10,2026-04-16T07:26:25.962Z
2022-02,East,5063232.50,97,52198.27,2026-04-16T07:26:25.962Z


In [0]:
sql_user = 'azure_user_1'
sql_password = 'admin@123'   # your actual password

jdbc_url = (
    "jdbc:sqlserver://salesdb-server-sravya.database.windows.net:1433;"
    "database=SalesDB-reporting;"
    "encrypt=true;"
    "trustServerCertificate=false;"
)

df_to_write.write \
    .format('jdbc') \
    .option('url', jdbc_url) \
    .option('dbtable', 'dbo.gold_monthly_revenue') \
    .option('user', sql_user) \
    .option('password', sql_password) \
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver') \
    .mode('overwrite') \
    .save()

print('Gold data written to Azure SQL successfully!')

Gold data written to Azure SQL successfully!
